# LedgerLens · QLoRA fine-tuning of Llama 3 8B on a free T4

Trains the extraction adapter with `ml/finetune/train_lora.py` (docs/layers/06-fine-tuning.md).

1. **Runtime → Change runtime type → T4 GPU.**
2. Accept the Llama 3 license at
   <https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct> and have a read token ready.
3. Build the data locally and upload `data/processed/finetune/` to Google Drive:
   `python -m ml.finetune.prepare_dataset --out data/processed/finetune`.

Checkpoints go to Drive, so a timed-out session resumes: run the cells again with the same
`RUN_ID`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Settings

In [ ]:
REPO_URL = "https://github.com/Sam15b/LedgerLens.git"
BRANCH = "main"
DRIVE_ROOT = "/content/drive/MyDrive/ledgerlens"
DATA_DIR = f"{DRIVE_ROOT}/finetune-data"  # the uploaded data/processed/finetune
CHECKPOINT_DIR = f"{DRIVE_ROOT}/finetune"  # checkpoints + adapter, per run id
RUN_ID = "dry-run"  # reuse an id to resume that run

## Drive, code and libraries

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import os

if not os.path.isdir("/content/LedgerLens"):
    !git clone --depth 1 --branch {BRANCH} {REPO_URL} /content/LedgerLens
%cd /content/LedgerLens
!git log --oneline -1

In [ ]:
# Only what training needs (requirements-ml.txt also lists the GAN / diffusion / Keras stacks).
!pip install -q "transformers>=4.44" "peft>=0.12" "bitsandbytes>=0.43" "accelerate>=0.33" pyyaml

In [ ]:
from huggingface_hub import notebook_login

notebook_login()  # a token that can read meta-llama/Meta-Llama-3-8B-Instruct

In [ ]:
!mkdir -p data/processed && rm -rf data/processed/finetune
!cp -r "{DATA_DIR}" data/processed/finetune
!wc -l data/processed/finetune/*.jsonl

## Dry run: 10 steps

Checks that the 4-bit model, the adapters, the data and checkpointing all work before a
long run. Done when the loss is logged, `checkpoint-10` and the adapter are on Drive and
`ml/finetune/runs/dry-run/run.json` says `finished`.

In [ ]:
!python -m ml.finetune.train_lora --config ml/finetune/configs/run.yaml \
    --set max_steps=10 --set logging_steps=1 --set save_steps=5 \
    --set run_id={RUN_ID} --set checkpoint_dir={CHECKPOINT_DIR}

In [ ]:
import json

with open(f"ml/finetune/runs/{RUN_ID}/run.json") as handle:
    run = json.load(handle)
print(run["status"], run["global_step"], run["metrics"])
!tail -n 3 ml/finetune/runs/{RUN_ID}/metrics.jsonl
!ls {CHECKPOINT_DIR}/{RUN_ID}

## Full run

Set `RUN_ID` (e.g. `run-1`) in the settings cell and run this. If the session ends, rerun
the setup cells and this one with the same id: it resumes from the last checkpoint.

In [ ]:
!python -m ml.finetune.train_lora --config ml/finetune/configs/run.yaml \
    --set run_id={RUN_ID} --set checkpoint_dir={CHECKPOINT_DIR}

## Keep the run log

`ml/finetune/runs/<run_id>/` holds JSON only (config, environment, data hashes, metrics).
Copy it to Drive, then commit it from your machine. The adapter weights stay on Drive.

In [ ]:
!mkdir -p "{DRIVE_ROOT}/runs" && cp -r ml/finetune/runs/{RUN_ID} "{DRIVE_ROOT}/runs/"
!ls "{DRIVE_ROOT}/runs/{RUN_ID}"